# PROJECT : HousePrice Prediction (from Kaggle)



###  LIBRARIES PURPOSE AND USE:

 Below is a brief explanation of each library and its purpose in the project.

---

## 1. pandas (`pd`)
- Used for data manipulation and analysis.
- Provides the DataFrame structure for storing tabular data.
- Helps in loading, cleaning, filtering, and exploring the dataset.
---


## 2. scikit-learn (`sklearn`)
This is the main machine learning library used in the project. It includes:

- `model_selection`: For splitting data into training and testing sets using `train_test_split()`.
- `svm`: Provides Support Vector Machine (SVM) models for regression and classification.
- `preprocessing`: Used for scaling and normalizing data using tools like `StandardScaler()`.
- `pipeline`: Helps combine preprocessing steps and model training into one workflow.

---
## 3. pickle
- A built-in Python module for saving and loading objects.
- Used to store the trained model in a `.pkl` file.
- Helps reuse the model later without retraining.
---




In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
import pickle


##  DATA COLLECTION & PREPROCESSING CLASS :

In [8]:
class Data:
    """
    Class: Data
    Purpose:
        - Step 1: Data Understanding
        - Step 2: Data Preprocessing
    """

    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None

    def load_data(self):
        """Load dataset from CSV file"""
        self.data = pd.read_csv(self.file_path)
        return self.data

    def head(self, n=5):
        """Return top rows of dataset"""
        return self.data.head(n)

    def missing_values(self):
        """Show count of missing values"""
        return self.data.isnull().sum()

    def duplicate_rows(self):
        """Count duplicate rows"""
        return self.data.duplicated().sum()

    def drop_columns(self, cols):
        """Drop unnecessary columns"""
        self.data.drop(columns=cols, inplace=True, errors="ignore")
        return self.data

    def fill_missing_mean(self):
        """Fill numeric missing values with mean"""
        self.data.fillna(self.data.mean(numeric_only=True), inplace=True)

    def fill_missing_mode(self):
        """Fill non-numeric missing values with mode"""
        self.data.fillna(self.data.mode().iloc[0], inplace=True)

    def encode_categorical(self):
        """Encode all categorical (object) columns"""
        self.data = pd.get_dummies(self.data, drop_first=True)
        return self.data

    def get_data(self):
        """Return processed dataset"""
        return self.data


###  
##  DATA SPLIT CLASS:

In [ ]:
class Model:
    """Split dataset into training and testing sets"""

    def __init__(self, data, target_column):
        self.data = data
        self.target_column = target_column

    def split(self, test_size=0.2, random_state=42):
        X = self.data.drop(columns=[self.target_column])
        y = self.data[self.target_column]
        return train_test_split(X, y, test_size=test_size, random_state=random_state)






X_train shape: (3680, 8)
X_test shape: (920, 8)



##  MODEL TRAIN CLASS

In [13]:
class Trainer:
    """Train SVM Regression Model"""

    def __init__(self, X_train, y_train):
        self.X_train = X_train
        self.y_train = y_train

    def train_svm(self, kernel="rbf", C=100, gamma=0.1):
        model = SVR(kernel=kernel, C=C, gamma=gamma)
        model.fit(self.X_train, self.y_train)
        return model


## PICKLE CLASS

In [18]:
class Store:
    """Save and Load Trained Model using Pickle"""

    def __init__(self, model):
        self.model = model

    def save(self, file_name):
        with open(file_name, "wb") as f:
            pickle.dump(self.model, f)
        print(f" Model saved as {file_name}")

    def load(self, file_name):
        with open(file_name, "rb") as f:
            loaded_model = pickle.load(f)
        print(f"Model loaded from {file_name}")
        return loaded_model


# APPLYING ALL STEPS :

## STEP 01 : DATA UNDERSTANDING &  PREPROCESSING 

In [10]:
# ===== Using Data Class =====
h = Data("houseprice.csv")
df = h.load_data()
print("First rows:\n", h.head())
print("Missing values:\n", h.missing_values())
print("Duplicates:", h.duplicate_rows())

# Drop unused columns (example)
h.drop_columns(["date", "street", "city", "statezip", "country"])

# Fill and Encode
h.fill_missing_mean()
h.fill_missing_mode()
h.encode_categorical()

clean_data = h.get_data()
print("Cleaned data shape:", clean_data.shape)

First rows:
             date      price  bedrooms  bathrooms  floors  waterfront  view  \
0  5/2/2014 0:00   313000.0         3       1.50     1.5           0     0   
1  5/2/2014 0:00  2384000.0         5       2.50     2.0           0     4   
2  5/2/2014 0:00   342000.0         3       2.00     1.0           0     0   
3  5/2/2014 0:00   420000.0         3       2.25     1.0           0     0   
4  5/2/2014 0:00   550000.0         4       2.50     1.0           0     0   

   condition  sqft_above  sqft_basement                    street       city  \
0          3        1340              0      18810 Densmore Ave N  Shoreline   
1          5        3370            280           709 W Blaine St    Seattle   
2          4        1930              0  26206-26214 143rd Ave SE       Kent   
3          4        1000           1000           857 170th Pl NE   Bellevue   
4          4        1140            800         9105 170th Ave NE    Redmond   

  country  
0     USA  
1     USA  
2

# STEP 02: DATA SPLITTING

In [12]:
splitter = Model(clean_data, target_column="price")
X_train, X_test, y_train, y_test = splitter.split()
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (3680, 8)
X_test shape: (920, 8)


# STEP 03 : MODEL TRAIN (SVM)

In [15]:

trainer = Trainer(X_train, y_train)
trained_model = trainer.train_svm()
print(" Model Training Complete")

 Model Training Complete


# STEP 04 : MODEL STORE IN PICKLE 

In [21]:
store_obj = Store(trained_model)
store_obj.save("svm_house_model.pkl")
loaded_model = store_obj.load("svm_house_model.pkl")



 Model saved as svm_house_model.pkl
Model loaded from svm_house_model.pkl


# STEP 05: MODEL PREDICTION :

In [20]:
user_input = {
    'bedrooms': 3,
    'bathrooms': 2,
    'sqft_living': 1800,
    'sqft_lot': 5000,
    'floors': 1,
    'waterfront': 0,
    'view': 0,
    'condition': 3,
    'grade': 7,
    'sqft_above': 1500,
    'sqft_basement': 300,
    'yr_built': 2005,
    'yr_renovated': 0,
    'zipcode_98103': 1  # Example encoded feature
}

input_df = pd.DataFrame([user_input])
model_columns = X_train.columns
input_sample = input_df.reindex(columns=model_columns, fill_value=0)
prediction = loaded_model.predict(input_sample)
print(f"Predicted House Price: ${prediction[0]:,.2f}")

Predicted House Price: $464,020.74
